# Week 04 — IP102 Training-Sample Scoring (E0)

**Research:** Investigating the Impact of Targeted Training Data Cleaning on Pest Classification Performance and Minority Classes in the IP102 Dataset.

**Goal:** Score **training images only** using the frozen, best-validation E0 ResNet50 checkpoint. Produce reproducible per-image loss, true-class confidence, prediction margin, entropy, and class-level summaries. **No training, deletion, or test-set evaluation happens here.**

> **Important methodological note:** For a fixed model, cross-entropy loss equals `-log(p_true)`. Therefore, selecting the highest-loss 5% and selecting the lowest-true-class-confidence 5% yields the same samples (up to ties). The originally proposed E2 and E3 are not independent interventions. Do not describe them as different cleaning methods. An alternative E3 based on **smallest true-class-vs-best-other-class margin** is provided as an *optional proposed protocol amendment*, not an automatically approved change.

**Requirements:** Run in Google Colab after making the IP102 dataset available at the same path used for Week 03. Keep E0's checkpoint in Google Drive; never commit it to GitHub.

## 1. Mount Drive and configure paths
The Week 03 dataset path was `/content/IP102_extracted/ip102_v1.1`. Colab storage resets between sessions: if it does not exist, repeat your Week 03 dataset download/extraction step **before** continuing.

In [ ]:
from pathlib import Path
import random, hashlib, json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from tqdm.auto import tqdm
from google.colab import drive

drive.mount('/content/drive')
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
DATA_DIR = Path('/content/IP102_extracted/ip102_v1.1')
IMAGE_DIR = DATA_DIR / 'images'
TRAIN_ANNOTATIONS = DATA_DIR / 'train.txt'
CHECKPOINT_PATH = Path('/content/drive/MyDrive/IP102_Proje/results/E0/best_model.pth')
OUTPUT_DIR = Path('/content/drive/MyDrive/IP102_Proje/results/week04')
BATCH_SIZE = 64 if DEVICE.type == 'cuda' else 16
NUM_WORKERS = 2
print('Device:', DEVICE)
print('Data directory:', DATA_DIR)
print('Checkpoint:', CHECKPOINT_PATH)
assert TRAIN_ANNOTATIONS.is_file(), ('Missing train.txt. Re-extract IP102 as in Week 03, '
                                      'or update DATA_DIR. Colab local storage is temporary.')
assert IMAGE_DIR.is_dir(), f'Missing image directory: {IMAGE_DIR}'
assert CHECKPOINT_PATH.is_file(), f'Missing E0 checkpoint: {CHECKPOINT_PATH}'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

## 2. Load and validate the official training split
The original Week 03 notebook used whitespace-delimited `train.txt` with `filename class_id`, and image files in `images/`. We preserve the exact annotation order for traceability.

In [ ]:
train_df = pd.read_csv(TRAIN_ANNOTATIONS, sep=r'\s+', header=None,
                       names=['filename', 'class_id'], dtype={'filename': str, 'class_id': int})
assert len(train_df) == 45095, f'Unexpected training size: {len(train_df)} (expected 45,095)'
assert train_df['class_id'].between(0, 101).all()
assert not train_df['filename'].duplicated().any(), 'Duplicate filenames in training annotations'
missing = [name for name in train_df['filename'] if not (IMAGE_DIR / name).is_file()]
assert not missing, f'{len(missing)} images missing; first examples: {missing[:5]}'
train_df['sample_index'] = np.arange(len(train_df), dtype=int)
print('Training images:', len(train_df), '| Classes:', train_df.class_id.nunique())
display(train_df.head())

## 3. Deterministic E0 evaluation preprocessing
**Do not use** the Week 03 random training augmentations for scoring. Use the same deterministic preprocessing as the Week 03 validation/test evaluation: Resize(256,256) → CenterCrop(224) → ImageNet normalization.

In [ ]:
eval_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

class IP102TrainScoringDataset(Dataset):
    def __init__(self, frame):
        self.frame = frame.reset_index(drop=True)
    def __len__(self):
        return len(self.frame)
    def __getitem__(self, index):
        row = self.frame.iloc[index]
        with Image.open(IMAGE_DIR / row.filename) as image:
            tensor = eval_transform(image.convert('RGB'))
        return tensor, int(row.class_id), int(row.sample_index)

loader = DataLoader(IP102TrainScoringDataset(train_df), batch_size=BATCH_SIZE,
                    shuffle=False, num_workers=NUM_WORKERS,
                    pin_memory=(DEVICE.type == 'cuda'))
print('Batches:', len(loader))

## 4. Load the frozen E0 checkpoint
We use the saved **best-validation** E0 checkpoint. No optimizer, gradients, or fine-tuning.

In [ ]:
model = models.resnet50(weights=None)
model.fc = nn.Linear(model.fc.in_features, 102)
checkpoint = torch.load(CHECKPOINT_PATH, map_location='cpu', weights_only=True)
if isinstance(checkpoint, dict) and 'model_state_dict' in checkpoint:
    state_dict = checkpoint['model_state_dict']
elif isinstance(checkpoint, dict) and 'state_dict' in checkpoint:
    state_dict = checkpoint['state_dict']
else:
    state_dict = checkpoint
model.load_state_dict(state_dict, strict=True)
model = model.to(DEVICE).eval()
for param in model.parameters(): param.requires_grad_(False)
print('Loaded E0 model. Checkpoint epoch:', checkpoint.get('epoch', 'unknown')
      if isinstance(checkpoint, dict) and 'model_state_dict' in checkpoint else 'unknown')

## 5. Score every training image
Outputs: per-image cross-entropy, true-class probability, top-1 prediction/confidence, correctness, entropy, and **margin = p(true) − max p(other classes)**. Negative margin indicates some competing class received a higher probability. We compute from full logits without rounding.

In [ ]:
records = []
with torch.inference_mode():
    for images, labels, indices in tqdm(loader, desc='Scoring E0 training images'):
        images = images.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)
        logits = model(images)
        probabilities = torch.softmax(logits, dim=1)
        losses = F.cross_entropy(logits, labels, reduction='none')
        true_probs = probabilities.gather(1, labels[:, None]).squeeze(1)
        top_probs, predictions = probabilities.max(dim=1)
        others = probabilities.clone()
        others.scatter_(1, labels[:, None], -1.0)
        best_other_probs = others.max(dim=1).values
        margins = true_probs - best_other_probs
        entropies = -(probabilities * probabilities.clamp_min(1e-12).log()).sum(dim=1)
        for j in range(labels.shape[0]):
            records.append({
                'sample_index': int(indices[j]),
                'class_id': int(labels[j]),
                'predicted_class_id': int(predictions[j]),
                'correct': bool(predictions[j] == labels[j]),
                'cross_entropy_loss': float(losses[j]),
                'true_class_confidence': float(true_probs[j]),
                'top1_confidence': float(top_probs[j]),
                'best_other_confidence': float(best_other_probs[j]),
                'true_vs_other_margin': float(margins[j]),
                'predictive_entropy': float(entropies[j]),
            })

scores_df = pd.DataFrame.from_records(records)
scores_df = train_df[['sample_index', 'filename', 'class_id']].merge(
    scores_df, on=['sample_index', 'class_id'], how='left', validate='one_to_one')
assert len(scores_df) == len(train_df) and not scores_df.isna().any().any()
assert np.allclose(scores_df.cross_entropy_loss.to_numpy(),
                   -np.log(scores_df.true_class_confidence.to_numpy()), atol=2e-4)
print('Scored:', len(scores_df))
print('Training-set top-1 accuracy (descriptive only):', round(scores_df.correct.mean(), 4))
display(scores_df.head())

## 6. Analyze class-level scores and minority classes
Tail classes are defined **a priori** from Week 02, not reselected based on model errors.

In [ ]:
TAIL_CLASS_IDS = [72, 80, 61, 63, 98, 81, 36, 35, 65, 17, 85,
                  12, 75, 60, 53, 55, 89, 43, 96, 90, 79]
class_summary = scores_df.groupby('class_id').agg(
    n_images=('filename', 'size'),
    mean_loss=('cross_entropy_loss', 'mean'),
    median_loss=('cross_entropy_loss', 'median'),
    mean_true_confidence=('true_class_confidence', 'mean'),
    train_top1_accuracy=('correct', 'mean'),
    mean_margin=('true_vs_other_margin', 'mean'),
).reset_index()
class_summary['is_tail'] = class_summary.class_id.isin(TAIL_CLASS_IDS)
assert len(class_summary) == 102
print('Tail images:', scores_df.class_id.isin(TAIL_CLASS_IDS).sum())
display(class_summary.sort_values('mean_loss', ascending=False).head(15))

## 7. Compare candidate removal rules (analysis only)
Set the removal budget to 5% of **training** images. Random E1 uses a fixed seed. E2 is highest cross-entropy loss. The originally proposed E3, lowest true-class confidence, is **mathematically redundant with E2**. We verify this rather than misrepresent it. A possible **amended** E3 uses lowest true-vs-best-other margin; changing the protocol requires documenting the amendment before training. E4 caps removal at `floor(0.05 × class_size)` for each class, so its actual total removal may be **less than 5%**.

In [ ]:
REMOVAL_FRACTION = 0.05
budget = int(np.floor(len(scores_df) * REMOVAL_FRACTION))
# Deterministic tie-breaking: sample_index.
e1_random = set(np.random.default_rng(SEED).choice(scores_df.sample_index.to_numpy(),
                                                   size=budget, replace=False).tolist())
e2_high_loss = set(scores_df.sort_values(
    ['cross_entropy_loss', 'sample_index'], ascending=[False, True]
).head(budget).sample_index.tolist())
e3_low_conf = set(scores_df.sort_values(
    ['true_class_confidence', 'sample_index'], ascending=[True, True]
).head(budget).sample_index.tolist())
e3_proposed_margin = set(scores_df.sort_values(
    ['true_vs_other_margin', 'sample_index'], ascending=[True, True]
).head(budget).sample_index.tolist())
# E4: select highest-loss samples within each class up to its own 5% floor.
e4_capped = set()
for class_id, group in scores_df.groupby('class_id'):
    class_cap = int(np.floor(REMOVAL_FRACTION * len(group)))
    chosen = group.sort_values(['cross_entropy_loss', 'sample_index'],
                               ascending=[False, True]).head(class_cap)
    e4_capped.update(chosen.sample_index.tolist())

print('Total 5% budget:', budget)
print('E1 random:', len(e1_random), '| E2 high-loss:', len(e2_high_loss),
      '| Original E3 low-confidence:', len(e3_low_conf),
      '| E4 per-class capped:', len(e4_capped))
print('E2 / original E3 overlap:', len(e2_high_loss & e3_low_conf), '/', budget)
assert e2_high_loss == e3_low_conf, ('Unexpected ranking mismatch: investigate numeric '
                                    'precision/ties before comparing E2 and E3.')
print('Proposed margin-based E3 overlap with E2:',
      len(e2_high_loss & e3_proposed_margin), '/', budget)

def summarize_removal(label, indices):
    selected = scores_df[scores_df.sample_index.isin(indices)]
    return {'rule': label, 'removed_n': len(selected),
            'removed_pct': round(100 * len(selected) / len(scores_df), 3),
            'tail_removed_n': int(selected.class_id.isin(TAIL_CLASS_IDS).sum()),
            'tail_removed_pct_of_tail': round(100 * selected.class_id.isin(TAIL_CLASS_IDS).sum() /
                                              scores_df.class_id.isin(TAIL_CLASS_IDS).sum(), 3)}
removal_summary = pd.DataFrame([
    summarize_removal('E1_random_5pct', e1_random),
    summarize_removal('E2_high_loss_5pct', e2_high_loss),
    summarize_removal('E3_original_low_confidence_5pct_DUPLICATE', e3_low_conf),
    summarize_removal('E3_proposed_low_margin_5pct_NOT_APPROVED', e3_proposed_margin),
    summarize_removal('E4_high_loss_per_class_5pct_cap', e4_capped),
])
display(removal_summary)

## 8. Export reproducible CSV files to Google Drive
All outputs are **analysis artifacts**; no training data is deleted. The proposed E3 margin set is labeled explicitly as a proposal. Store CSVs in GitHub later; keep the large model checkpoint in Drive.

In [ ]:
scores_df.to_csv(OUTPUT_DIR / 'training_sample_scores.csv', index=False, float_format='%.10g')
class_summary.to_csv(OUTPUT_DIR / 'class_score_summary.csv', index=False, float_format='%.10g')
removal_summary.to_csv(OUTPUT_DIR / 'removal_rule_summary.csv', index=False)

selection_sets = {
    'E1_random_5pct': e1_random,
    'E2_high_loss_5pct': e2_high_loss,
    'E3_original_low_confidence_5pct_DUPLICATE': e3_low_conf,
    'E3_proposed_low_margin_5pct_NOT_APPROVED': e3_proposed_margin,
    'E4_high_loss_per_class_5pct_cap': e4_capped,
}
selection_df = scores_df[['sample_index', 'filename', 'class_id']].copy()
for label, selected in selection_sets.items():
    selection_df[label] = selection_df.sample_index.isin(selected)
selection_df.to_csv(OUTPUT_DIR / 'candidate_removal_flags.csv', index=False)

metadata = {
    'checkpoint_path': str(CHECKPOINT_PATH),
    'checkpoint_sha256': hashlib.sha256(CHECKPOINT_PATH.read_bytes()).hexdigest(),
    'annotation_path': str(TRAIN_ANNOTATIONS),
    'annotation_sha256': hashlib.sha256(TRAIN_ANNOTATIONS.read_bytes()).hexdigest(),
    'num_training_images': int(len(scores_df)),
    'seed': SEED,
    'scoring_transform': 'Resize((256,256)); CenterCrop(224); ToTensor; ImageNet Normalize',
    'budget_rule': 'floor(0.05 * total training images)',
    'e4_cap_rule': 'floor(0.05 * per-class training images)',
    'protocol_warning': 'Original E3 low true-class confidence duplicates E2 high cross-entropy loss.',
    'e3_margin_status': 'proposed amendment only; do not treat as approved protocol',
}
(OUTPUT_DIR / 'scoring_metadata.json').write_text(json.dumps(metadata, indent=2), encoding='utf-8')
print('Saved outputs to:', OUTPUT_DIR)
for file in sorted(OUTPUT_DIR.iterdir()):
    if file.is_file(): print(' -', file.name, f'({file.stat().st_size:,} bytes)')

## 9. Optional: download outputs to your Mac
Files are already saved to Google Drive. Run this cell if you want to download them for `results/week04/` in the GitHub repository. Do **not** upload the checkpoint.

In [ ]:
# Set to True only when you want a ZIP download.
DOWNLOAD_RESULTS_ZIP = False
if DOWNLOAD_RESULTS_ZIP:
    import shutil
    from google.colab import files
    zip_path = shutil.make_archive('/content/week04_scoring_results', 'zip', root_dir=OUTPUT_DIR)
    files.download(zip_path)
else:
    print('Outputs are safely stored in Drive:', OUTPUT_DIR)

## 10. Interpretation and next steps

1. Inspect the **high-loss** examples manually before calling them label noise: high loss may reflect rare appearances, domain shift, or intrinsically difficult but correctly labeled images.
2. Check how often global 5% removal targets tail classes compared with E4's class cap.
3. **Resolve the E2/E3 duplication in the written protocol before Week 05.** If using margin-based E3, record the amendment and explain why it tests a different ranking.
4. Keep the official validation/test sets untouched. Evaluate E1–E4 with the same training hyperparameters and model-selection criteria as E0.
5. When committing results, review the files and ensure no model weights, private photos, or unnecessary outputs are included.

**Suggested GitHub outputs:** `notebooks/week04_training_sample_scoring.ipynb`, `results/week04/*.csv`, `reports/week04_scoring_report.md` (report written after inspecting results).